<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/22_context_eval_frozen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/22_context_eval_frozen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 22 — EVAL-016 Aşama 1: dondurulmuş aile için bağlam değerlendirmesi (CPU)

**Bilimsel amaç** (D-023, D-025, D-037; plan `docs/EVAL016_PLAN.md`): ses, ölçülen kayıt bağlamı (saat, tarih) ve yaşın ötesinde astım bilgisi taşıyor mu? EXP-016'nın **kaydedilmiş** tahminleri ve gömmeleri üzerinde, sonuçlardan önce yazılmış testler. Yeniden eğitim yok, GPU yok.

- **Birincil (L2, T1):** `y ~ cr(saat, 4) + cr(tarih, 4) + yaş + ŝ`, ŝ için Firth cezalı olabilirlik oranı testi; 6 backbone Holm.
- **Destekleyiciler:**
  - bootstrap OR; gün-kümeli jackknife (CV3; C1 koşul 3) ve gün-kümeli sandviç (CR1; betimsel); spline df 6;
  - E2h-KLR; tabakalı AUC'ler ve referans çizgileri;
  - Spisak tam testi;
  - etiket-içi permütasyon kısmi testi (C2);
  - karşıt hücreler; N2;
  - G1 bağlam probu ve N5;
  - codec zinciri; oynaklık;
  - kalibrasyon tanılaması (gerçek bağlam + sinyalsiz yapay skor; K2 geçerlilik kuralı buradan değerlendirilir).
- **Kararlar:** önceden yazılmış C1 / C2 kurallarıyla. **N3 Aşama 2'dedir;** bu notebook C1'i hiçbir backbone için tamamlamaz.

**Çalışma ortamı:** *Runtime → Change runtime type → CPU* yeterli. Süre ~40–80 dk [tahmin: simüle Colab'da (2 CPU) 195 sentetik katılımcıyla 26 dk ölçüldü; gerçek kohort ~1.7 kat büyük]. *Run all* kullanılabilir; bir adım başarısız olursa notebook durur.

**Onay kapısı (D-038):** uygulama protokolü v1.2 KABUL (Firth PLR; etiket-içi permütasyon kısmi testi; C1 koşul 3 = gün-kümeli jackknife CV3; N5 betimsel). Analiz ayarları sabittir (bootstrap 2000, permütasyon 1000, kalibrasyon 500 × 3 senaryo, G1 5 tekrar, tohum 0, 6 backbone, spline df 4). Script, onaysız protokolle ya da farklı bir ayarla gerçek veride çalışmayı reddeder. **Bu notebook'ta ayar değiştirme;** komut satırına argüman ekleme.

**Yorum kuralı:** Sonuçlar raporun 7–8. bölümlerindeki önceden yazılmış kurallarla okunur. "Gösterilemedi", "yok" demek değildir; pozitif bir sonuç da klinik tanı aracı anlamına gelmez. Raporu birlikte inceleyelim; Aşama 2 ve Faz 3 tasarımına ondan sonra geçeriz.

## 0. Oturum kurulumu

In [1]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bu notebook CPU üzerinde çalışır; sorun değil.')

Python: 3.13.16 | Linux-6.6.122+-x86_64-with-glibc2.39
UYARI: GPU yok (CPU oturumu). Bu notebook CPU üzerinde çalışır; sorun değil.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
BACKBONES   = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

Girdiler mevcut.


In [4]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

Klonlandı.
Dal: main | commit: 4670094c9ada (= origin/main)


In [5]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
# mlconfound 0.21.3: pkg_resources ister (setuptools<81); gam yolu güncel scipy ile çalışmıyor → 'linear' (EVAL016_PLAN S3)
sh('pip install -q mlconfound==0.21.3 "setuptools<81"')
import numpy as np, pandas as pd, sklearn, statsmodels, scipy, patsy
import mlconfound.stats  # kurulum doğrulaması
print('numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__, '| statsmodels', statsmodels.__version__,
      '| scipy', scipy.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.3 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/mlconfound/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1 | statsmodels 0.15.0 | scipy 1.16.3


In [6]:
# Kapı: plan ve EXP-016 main'de; EXP-016 ara sonuçları (8 kol × 5 tekrar) ve gömmeler Drive'da; önbellek index'i doğrulanmış
for p in ['docs/EVAL016_PLAN.md', 'reports/frozen/EXP-016_frozen.json', 'scripts/evaluate_context.py']:
    if git('ls-files', '--error-unmatch', p, check=False).returncode != 0:
        raise RuntimeError(f'{p} main dalında yok: ilgili PR birleştirilmiş mi? Sonra 0. bölümü yeniden çalıştır.')
PROBE_DIR = EXPERIMENTS / 'frozen_probe'
ARMS = BACKBONES + ['mfcc_lr', 'mfcc_mlp']
missing = [f'{a}_r{r}' for a in ARMS for r in range(5)
           if not ((PROBE_DIR / 'partial' / f'{a}_r{r}_test.csv').exists() and (PROBE_DIR / 'partial' / f'{a}_r{r}.json').exists())]
assert not missing, f'EXP-016 ara sonuçları eksik: {missing[:5]} … (notebook 21 EXP-016)'
for b in BACKBONES:
    assert (EMB_DIR / b / 'MANIFEST.sha256').exists(), f'{b}: gömmeler yok'
cache_info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
assert sha256(DRIVE_CACHE / 'index.csv') == cache_info['files_sha256']['index.csv'], 'önbellek index.csv doğrulanamadı'
CONTEXT_OUT = EXPERIMENTS / 'context_eval'
# Protokol: KABUL ve D-038 main'de (onay PR'ı birleştirilmiş); sabit ayarlar script'ten okunur (yeni süreçte, önbelleksiz)
if '## D-038' not in (REPO_DIR / 'DECISIONS.md').read_text():
    raise RuntimeError('D-038 main dalında yok: EVAL-016 protokol onay PR\'ı birleştirilmiş mi? Sonra 0. bölümü yeniden çalıştır.')
r = subprocess.run([sys.executable, '-c', 'import sys, json; sys.path.insert(0, "scripts"); import evaluate_context as E; '
                    'print(json.dumps({"protocol": E.PROTOCOL, "fixed": E.FIXED}, ensure_ascii=False))'],
                   cwd=REPO_DIR, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
PROT = json.loads(r.stdout.strip().splitlines()[-1])
assert PROT['protocol']['status'] == 'KABUL', f"protokol onaylı değil: {PROT['protocol']['version']} / {PROT['protocol']['status']}"
print('Protokol:', PROT['protocol']['version'], '|', PROT['protocol']['status'], '| C1 koşul 3:', PROT['protocol']['c1_cond3'])
print('Sabit ayarlar:', PROT['fixed'])
print('Kapı geçti: plan, D-038, EXP-016 ara sonuçları (8 kol × 5 tekrar), gömmeler ve önbellek index\'i hazır.')

Protokol: EVAL-016 uygulama v1.2 (D-038) | KABUL | C1 koşul 3: cluster_cv3
Sabit ayarlar: {'n_repeats': 5, 'n_boot': 2000, 'n_perms': 1000, 'n_sim': 500, 'g1_repeats': 5, 'seed': 0, 'backbones': 'cnn10,cnn14,cnn14_16k,beats,wavlm_base_plus,wavlm_large', 'spline_df': 4}
Kapı geçti: plan, D-038, EXP-016 ara sonuçları (8 kol × 5 tekrar), gömmeler ve önbellek index'i hazır.


## 1. EVAL-016 Aşama 1
Katılımcı düzeyi tablo Drive `experiments/context_eval/` altına; git'e yalnız agrega rapor (`reports/context/EVAL-016_context.{md,json}`) yazılır. Script, raporda katılımcı ID'si bulursa yazmayı reddeder (D-014).

In [7]:
sh(f'cd {REPO_DIR} && python scripts/evaluate_context.py --probe-dir {PROBE_DIR} --context {DERIVED}/participant_context.csv '
   f'--splits-dir {DERIVED}/splits --emb-root {EMB_DIR} --cache-index {DRIVE_CACHE}/index.csv '
   f'--out-dir {CONTEXT_OUT} --report-dir {REPO_DIR}/reports/context')

kohort: {'n_cohort': 342, 'n_t1': 339, 'n_t1_asthma': 282, 'n_t1_healthy': 57, 'n_excluded': 3, 'excluded_by_label': {'astım': 1, 'sağlıklı': 2}, 'n_days': 65}
kalibrasyon tanılaması (500 × 3 senaryo; PLR, CR1, CV3) …
  K2 geçerlilik: {'valid': True, 'rule': "kalibrasyon tanılamasında CV3'ün iki yönlü p < 0.05 ret oranının Wilson %95 alt sınırı S1 ve S3'te ≤ 0.075 (Bradley'nin liberal ölçütü, 1.5α); değilse koşul 3 geçersiz → C1 'kurulamadı (K2)'", 'cv3_wilson_lower': {'S1_smooth': 0.0152562373229252, 'S3_day': 0.022890872735181415}, 'threshold': 0.075}
T1 …
  cnn10: LR 12.35 p 0.0004 CV3 p 0.0028 OR 3.06 [1.80, 9.67]
  cnn14: LR 7.50 p 0.0062 CV3 p 0.0151 OR 2.14 [1.30, 5.11]
  cnn14_16k: LR 2.13 p 0.1447 CV3 p 0.1916 OR 1.55 [1.01, 2.88]
  beats: LR 17.58 p 0.0000 CV3 p 0.0202 OR 4.69 [2.66, 18.76]
  wavlm_base_plus: LR 9.49 p 0.0021 CV3 p 0.0677 OR 2.75 [1.45, 9.39]
  wavlm_large: LR 15.83 p 0.0001 CV3 p 0.0167 OR 3.83 [1.97, 14.73]
  mfcc_lr: LR 6.82 p 0.0090 CV3 p 0.1029 OR 2.10 [

## 2. Önceden yazılmış kurallara göre özet (yorum yok; rapor birlikte incelenecek)

In [8]:
rep = json.loads((REPO_DIR / 'reports/context/EVAL-016_context.json').read_text())
c = rep['cohort']
print(f"T1 kohortu: {c['n_t1']} ({c['n_t1_asthma']} astım / {c['n_t1_healthy']} sağlıklı); dışlanan {c['n_excluded']} {c['excluded_by_label']}")
cal = rep['calibration']
print('Çalıştırma kipi:', rep['run_mode'], '| protokol', rep['protocol']['version'], rep['protocol']['status'])
print('Kalibrasyon (sinyalsiz, p<0.05) PLR / CR1 / CV3:',
      {k: (cal[k]['reject_05'], cal[k]['reject_05_cluster'], cal[k]['reject_05_cv3']) for k in ('S1_smooth', 'S2_step', 'S3_day')})
print('K2 geçerlilik (önceden yazılmış kural):', cal['k2_validity']['valid'], cal['k2_validity']['cv3_wilson_lower'])
rows = []
for b in rep['backbones']:
    v, t = rep['verdicts'][b], rep['t1'][b]
    rows.append({'backbone': b, 'T1 PLR p': t['p_lrt'], 'Holm p': rep['holm'][b], 'OR/SD': t['or_sd'], 'CV3 p': t['p_cv3'],
                 'boot CI': (rep['boot'][b]['lo'], rep['boot'][b]['hi']), 'G1+': v['g1_positive'],
                 'Aşama-1 koşulları': v['stage1_all_met'], 'C1': v['c1_status'], 'Aşama 2': v['stage2_triggered'], 'C2': rep['c2'][b]})
pd.DataFrame(rows)

T1 kohortu: 339 (282 astım / 57 sağlıklı); dışlanan 3 {'astım': 1, 'sağlıklı': 2}
Çalıştırma kipi: gerçek veri kipi (protokol KABUL, ayarlar sabit) | protokol EVAL-016 uygulama v1.2 (D-038) KABUL
Kalibrasyon (sinyalsiz, p<0.05) PLR / CR1 / CV3: {'S1_smooth': (0.046, 0.102, 0.026), 'S2_step': (0.044, 0.104, 0.026), 'S3_day': (0.148, 0.162, 0.036)}
K2 geçerlilik (önceden yazılmış kural): True {'S1_smooth': 0.0152562373229252, 'S3_day': 0.022890872735181415}


,backbone,T1 PLR p,Holm p,OR/SD,CV3 p,boot CI,G1+,Aşama-1 koşulları,C1,Aşama 2,C2
0,cnn10,0.000441,0.001763,3.064202,0.002755,"(1.8040149711998017, 9.67089629384631)",False,True,Aşama 2 bekliyor (N3),True,gösterilemedi
1,cnn14,0.006170,0.012339,2.144891,0.015125,"(1.2985285626640066, 5.113502570419056)",False,True,Aşama 2 bekliyor (N3),True,gösterilemedi
2,cnn14_16k,0.144725,0.144725,1.551472,0.191590,"(1.0077725929513963, 2.8765354791743385)",False,False,sağlanmadı,False,gösterilemedi
3,beats,0.000028,0.000166,4.690567,0.020250,"(2.6638792386946637, 18.760067396319275)",False,True,Aşama 2 bekliyor (N3),True,gösterilemedi
4,wavlm_base_plus,0.002069,0.006207,2.747159,0.067739,"(1.445527687390746, 9.39408270450987)",False,False,sağlanmadı,True,gösterilemedi
5,wavlm_large,0.000069,0.000347,3.834091,0.016694,"(1.971479370975296, 14.729816336278573)",False,True,Aşama 2 bekliyor (N3),True,gösterilemedi


## 3. Raporu GitHub'a gönder (yeni dal)
Yalnız `reports/context/` commit edilir (gönderimden önce katılımcı ID'si ve token taraması). Notebook'u kaydetmeden önce çıktıları temizle.

In [9]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'eval016-results', ['reports/context'], 'EVAL-016 Aşama 1: dondurulmuş aile bağlam değerlendirmesi (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

commit: 5233855 | dosyalar: ['reports/context/EVAL-016_context.json', 'reports/context/EVAL-016_context.md']
branch 'eval016-results-20261010-223144' set up to track 'origin/eval016-results-20261010-223144'.
remote: 
remote: Create a pull request for 'eval016-results-20261010-223144' on GitHub by visiting:        
remote:      https://github.com/alperarslan19/asthma-voice-dl/pull/new/eval016-results-20261010-223144        
remote: 
To https://github.com/alperarslan19/asthma-voice-dl.git
 * [new branch]      eval016-results-20261010-223144 -> eval016-results-20261010-223144
Gönderildi: eval016-results-20261010-223144
PR aç: https://github.com/alperarslan19/asthma-voice-dl/pull/new/eval016-results-20261010-223144
